## **Notebook để preprocessing, chia train/val/test để sử dụng chung**



In [ ]:
pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import re, unicodedata, hashlib
import pandas as pd
from datasets import load_dataset, Dataset
import os
import numpy as np

In [ ]:
# Tải bộ dữ liệu 'hiimbach/mtet' từ Hugging Face Datasets.
dataset = load_dataset("hiimbach/mtet")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/250M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/250M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/251M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/83.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3750006 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/416668 [00:00<?, ? examples/s]

In [ ]:
# Định nghĩa các hằng số và tham số cấu hình cho việc chia dữ liệu và lưu trữ.
seed     = 42
n_train  = 230000     # số câu train ban đầu
n_valid   = 15000
n_test   = 25000
reverse   = 1200000    # số dòng train gốc xử lý sẵn, pool đủ lớn để mở rộng sau
out_dir   = "mtet260k_dataset"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# Tạo DataFrame từ tập huấn luyện và kiểm tra, sau đó xáo trộn và cắt bớt dữ liệu.
df_train = pd.DataFrame(dataset["train"].shuffle(seed = seed)).iloc[:reverse].reset_index(drop=True)
df_test  = pd.DataFrame(dataset["test"].shuffle(seed= seed)).reset_index(drop=True)
print(f"train gốc (đã cắt {reverse:,}): {len(df_train):,} | test gốc: {len(df_test):,}")

train gốc (đã cắt 1,200,000): 1,200,000 | test gốc: 416,668


In [ ]:
# Định nghĩa các biểu thức chính quy để làm sạch văn bản.
_re_url = re.compile(r"http\S+|www\S+|<.*?>")
_re_ws  = re.compile(r"\s+")

In [ ]:
# Hàm để chuẩn hóa và làm sạch văn bản.
def clean_series(s):
    s = s.fillna("").astype(str)
    s = s.map(lambda x: unicodedata.normalize("NFC", x))
    s = s.str.replace(_re_url, "", regex=True)
    s = s.str.replace(_re_ws, " ", regex=True).str.strip()
    return s

In [ ]:
# Áp dụng hàm làm sạch cho các cột 'vi' và 'en' của cả hai DataFrame.
for d in (df_train, df_test):
    d["vi"] = clean_series(d["vi"])
    d["en"] = clean_series(d["en"])


# hàm để lọc các cặp câu dựa trên độ dài và tỷ lệ.
def filter_quality(d):
    vl = d["vi"].str.split().str.len().to_numpy()
    el = d["en"].str.split().str.len().to_numpy()
    vl = np.nan_to_num(vl).astype(int)
    el = np.nan_to_num(el).astype(int)
    hi = np.maximum(vl, el)
    lo = np.minimum(vl, el).clip(min=1)          # tránh chia 0
    mask = (
        (d["vi"].str.len().to_numpy() > 0) &
        (d["en"].str.len().to_numpy() > 0) &
        (vl >= 3) & (vl <= 80) &
        (el >= 3) & (el <= 80) &
        (hi / lo <= 2.0)) # tỷ lệ độ dài không quá 2.0

    return d[mask].copy()

In [ ]:
# Lọc.
df_train = filter_quality(df_train)
df_test  = filter_quality(df_test)
print(f"sau lọc -> train {len(df_train):,} | test {len(df_test):,}")

sau lọc -> train 1,078,327 | test 374,452


In [ ]:
# Dedup lọc duplicate.
def dedup(d):
    d = d.drop_duplicates(subset=["en", "vi"])
    d = d.drop_duplicates(subset=["en"])
    d = d.drop_duplicates(subset=["vi"])
    return d.reset_index(drop=True)

df_train = dedup(df_train)
df_test  = dedup(df_test)
print(f"sau dedup -> train {len(df_train):,} | test {len(df_test):,}")

sau dedup -> train 1,074,303 | test 373,852


In [ ]:
# Chia tập kiểm tra thành tập xác thực (valid) và tập kiểm tra (test) cuối cùng.
need = n_valid + n_test

valid_df = df_test.iloc[:n_valid][["vi", "en"]].reset_index(drop=True)
test_df  = df_test.iloc[n_valid:n_valid + n_test][["vi", "en"]].reset_index(drop=True)
print(f">> valid {len(valid_df):,} | test {len(test_df):,}")

>> valid 15,000 | test 25,000


In [ ]:
# Loại bỏ các bản ghi trùng lặp giữa tập train và các tập val/test.
# loại overlapp
holdout_en = set(valid_df["en"]) | set(test_df["en"])
holdout_vi = set(valid_df["vi"]) | set(test_df["vi"])
before = len(df_train)
df_train = df_train[~df_train["en"].isin(holdout_en) & ~df_train["vi"].isin(holdout_vi)].reset_index(drop=True)
print(f"Loại overlap train: {before:,} -> {len(df_train):,}")

Loại overlap train: 1,074,303 -> 1,074,021


In [ ]:
# Trộn tập huấn luyện, sau đó chia thành tập huấn luyện chính và tập dữ liệu dự trữ.
df_train = df_train.sample(frac=1, random_state = seed).reset_index(drop=True)
train_df = df_train.iloc[:n_train][["vi", "en"]].reset_index(drop=True)
pool_df  = df_train.iloc[n_train:][["vi", "en"]].reset_index(drop=True)
print(f"train {len(train_df):,} | pool dự trữ {len(pool_df):,}")

train 230,000 | pool dự trữ 844,021


In [ ]:
additional_samples = 30000

train_df = pd.concat([train_df, pool_df.iloc[:additional_samples]], ignore_index=True)
pool_df = pool_df.iloc[additional_samples:].reset_index(drop=True)

print(f"train sau khi bổ sung: {len(train_df):,} | pool dự trữ còn lại: {len(pool_df):,}")

train sau khi bổ sung: 260,000 | pool dự trữ còn lại: 814,021


In [ ]:
# Lưu các DataFrame đã xử lý vào các tệp Parquet trong thư mục đầu ra.
train_df.to_parquet(f"{out_dir}/train.parquet", index=False)
valid_df.to_parquet(f"{out_dir}/valid.parquet", index=False)
test_df.to_parquet(f"{out_dir}/test.parquet",   index=False)
pool_df.to_parquet(f"{out_dir}/pool.parquet",   index=False)

In [ ]:
# Nén thư mục chứa các tệp Parquet và tải xuống.
import shutil
from google.colab import files

zip_filename = f"{out_dir}.zip"
shutil.make_archive(out_dir, 'zip', out_dir)

print(f"'{{zip_filename}}' đã được tạo. Đang tải xuống...")
files.download(zip_filename)


'{zip_filename}' đã được tạo. Đang tải xuống...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>